# Exam Project: The "Cost of Quality" 
## Legislative Shocks to Child Labor and the Fertility Transition in 19th-Century Britain

**Course:** Economics-Demography  
**Student ID:** LPX972
**Methodology:** Difference-in-Differences (DiD)  
**Data Sources:** Populations Past (1851–1891), UK Data Service (1831 Baseline)

---

### **1. Introduction & Motivation**
This project investigates the impact of the **1833 and 1844 Factory Acts** on the British fertility transition. While much of the literature focuses on the late-19th-century decline, this study explores whether earlier legislative restrictions on child labor and the introduction of mandatory schooling for factory children acted as a catalyst for the "Quantity-Quality" (Q-Q) trade-off.

### **2. Theoretical Framework**
The core of this analysis rests on the **Standard Quantity-Quality Model**. According to Unified Growth Theory, the primary trigger for the demographic transition is the rise in the demand for human capital. 

The **1833 Factory Act** introduced two critical economic shocks:
1. **The Opportunity Cost Shock**: By restricting the hours children aged 9–13 could work, it reduced the economic "benefit" of high-quantity fertility.
2. **The Schooling Mandate**: It required two hours of daily elementary education for factory children, effectively raising the "price" of child quality.

I test whether these shocks led to a faster decline in the **Total Fertility Rate (TFR)** in textile-heavy districts compared to agricultural controls.

### **3. Data and Variables**
The primary dataset is a Registration District (RD) level panel derived from **Populations Past**.

**Core Variables:**
* **`TFR` (Total Fertility Rate)**: The outcome variable measuring the intensive margin of fertility.
* **`F_TEX` / `SC6`**: Percentage of the population in the textile industry, used to identify the "Treatment" intensity of the Factory Acts.
* **`F_CL_1013`**: Specifically measures female child labor (ages 10–13), providing a direct mechanism for the labor restriction impact.
* **`IMR` (Infant Mortality Rate)**: Included as a control to account for "replacement fertility" effects.

### **4. Empirical Strategy**
I utilize a **Difference-in-Differences (DiD)** approach. The identification strategy relies on the staggered enforcement of factory laws by the professional **Factory Inspectorate** established in 1833.

**The Regression Model:**
$$Fertility_{it} = \alpha + \beta(Textile\_Intensity_i \times Post\_1833_t) + \Gamma X_{it} + \mu_i + \tau_t + \epsilon_{it}$$

* **$\beta$** is the coefficient of interest, representing the causal effect of the laws on textile districts relative to the baseline.
* **$X_{it}$** includes controls for infant mortality and population density.
* **$\mu_i$ and $\tau_t$** are district and year fixed effects to control for time-invariant local characteristics and national shocks.

In [13]:
import os
import sys
from pathlib import Path

# Add root project directory to path for modular imports
notebook_path = Path.cwd()
root_dir = notebook_path if (notebook_path / 'src').exists() else notebook_path.parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from src.utils import plotting, econometrics


In [ ]:
### DATA INTEGRITY CHECK
def check_integrity(df):
    print("--- Data Integrity Audit ---")
    print(f"Total Observations: {len(df)}")
    print(f"Missing TFR: {df['TFR'].isna().sum()}")
    print(f"Missing Textiles: {df['F_TEX'].isna().sum()}")
    print(f"Duplicate REGDIST/Year: {df.duplicated(subset=['REGDIST', 'Year']).sum()}")
    if df['TFR'].max() > 15:
        print("WARNING: Potential outliers in TFR detected!")

# Call check_integrity after df_panel is loaded
# check_integrity(df_panel)

In [14]:
# Load Spatial Data
geojson_path = root_dir / 'data' / 'raw' / 'data1851_0.geojson'
map_df = gpd.read_file(geojson_path)

# Visualize with modular tool
plotting.plot_map(map_df, 'TFR', 'Total Fertility Rate in 1851 by Registration District')
plt.show()

In [15]:
print(map_df.crs)

In [16]:
# Reproject to the British National Grid
map_df = map_df.to_crs(epsg=27700)

# Plot again to see the difference in shape
map_df.plot(figsize=(10, 12))

In [17]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 15))

# We'll plot TFR (Total Fertility Rate)
map_df.plot(column='TFR', 
            ax=ax, 
            legend=True, 
            cmap='YlOrRd', # Yellow to Orange to Red
            legend_kwds={'label': "Total Fertility Rate (1851)",
                         'orientation': "horizontal"})

ax.set_axis_off()
plt.title('Fertility Rates across Great Britain, 1851', fontsize=15)
plt.show()

In [18]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# 1. Setup figure - wider (16) to make room for the legend on the side
fig, ax = plt.subplots(1, 1, figsize=(16, 16), facecolor='#fdfcf0')
ax.set_facecolor('#fdfcf0')

# 2. Plotting with a relocated legend
map_df.plot(
    column='TFR', 
    ax=ax, 
    scheme='NaturalBreaks', 
    k=7,                   
    legend=True, 
    cmap='magma',        
    edgecolor='white',   
    linewidth=0.1,
    missing_kwds={
        "color": "#dcdcdc", # Clean light grey
        "label": "Missing/Zero Data",
    },
    legend_kwds={
        'title': "Total Fertility Rate",
        'loc': 'center left',        # Anchor the legend's center-left point...
        'bbox_to_anchor': (1, 0.5),  # ...at the very right edge of the map (1, 0.5)
        'fmt': "{:.1f}",   
        'frameon': False,
        'fontsize': 12
    }
)

# 3. Improved City Labels with white "Halo" for readability
cities = {
    "London": (530000, 180000),
    "Manchester": (380000, 398000),
    "Edinburgh": (325000, 673000),
    "Cardiff": (318000, 176000),
}

for city, pos in cities.items():
    # Add a small dot for the city location
    ax.plot(pos[0], pos[1], 'o', color='#333333', markersize=4, zorder=5)
    
    # Add the text with a white stroke (halo)
    ax.text(
        pos[0], pos[1] + 8000, city, # Shift text up slightly so it's not on the dot
        fontsize=14, 
        fontweight='bold', 
        ha='center',
        color='#333333',
        zorder=6,
        path_effects=[pe.withStroke(linewidth=3, foreground="white")]
    )

# 4. Cleanup
ax.axis('off')
plt.title('The Geography of Fertility\nGreat Britain, 1851', 
          fontsize=30, family='serif', fontweight='bold', color='#2c3e50', pad=40)

# This prevents the legend from being cut off since we moved it outside
plt.tight_layout()
plt.show()

In [21]:
### 14. Geospatial Evolution of Fertility (1851-1881)
# We visualize the Total Fertility Rate (TFR) across England and Wales for all four census waves.
# This helps identify regional clusters and the spatial diffusion of the fertility transition.

if 'map_df' in locals() and 'df_panel' in locals():
    # 1. Clean panel data for the required variables to avoid mismatches
    plot_vars = ['REGDIST', 'Year', 'TFR']
    df_map_data = df_panel[plot_vars].copy()

    # 2. Get year waves
    years = sorted(df_map_data['Year'].unique())
    
    # 3. Create a 2x2 multi-panel figure
    fig, axes = plt.subplots(2, 2, figsize=(20, 24))
    axes = axes.flatten()
    
    # Use a unified color scale (vmin/vmax) to make the transition comparable across time
    vmin = df_map_data['TFR'].min()
    vmax = df_map_data['TFR'].max()
    
    for i, year in enumerate(years):
        # Filter data for the specific year
        year_subset = df_map_data[df_map_data['Year'] == year]
        
        # Join the panel data to the geospatial boundaries
        # We use a left join on map_df to ensure all districts are drawn
        merged = map_df.merge(year_subset, on='REGDIST', how='left')
        
        # Plotting with 'magma' colormap (high values are bright yellow/white, low are dark purple/black)
        merged.plot(column='TFR', 
                    ax=axes[i], 
                    cmap='magma', 
                    vmin=vmin, 
                    vmax=vmax,
                    edgecolor='black', 
                    linewidth=0.1,
                    missing_kwds={'color': 'lightgrey'}) # Areas with missing data in grey
        
        axes[i].set_title(f'Total Fertility Rate in {year}', fontsize=18, fontweight='bold')
        axes[i].axis('off')
    
    # Add a single colorbar for the whole figure to represent the transition
    sm = plt.cm.ScalarMappable(cmap='magma', norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm._A = []
    cbar = fig.colorbar(sm, ax=axes, shrink=0.5, aspect=20, orientation='vertical', pad=0.05)
    cbar.set_label('Total Fertility Rate (TFR)', fontsize=14)
    
    plt.suptitle("Figure 14.1: The Spatial Diffusion of the British Fertility Transition (1851-1881)", 
                 fontsize=24, y=0.95, fontweight='bold')
    plt.show()
    
    print("Geospatial visualization complete. Notice the transition from yellow/orange (high fertility) "
          "to darker purple (lower fertility), especially in industrial areas and the south.")
else:
    print("ERROR: map_df or df_panel not found in memory. Please run cells 1-7 first.")

In [20]:
### LOAD PROCESSED MASTER PANEL DATA
import pandas as pd
from pathlib import Path

processed_data_path = Path('../data/processed/master_panel_data.csv')
if not processed_data_path.exists():
    processed_data_path = Path('exam_project/data/processed/master_panel_data.csv')

print(f"Loading data from: {processed_data_path}")
df_panel = pd.read_csv(processed_data_path)
df_factory = df_panel
print(f"Loaded {len(df_panel)} rows with {len(df_panel.columns)} columns.")
df_panel.head()


In [24]:
# Create a summary table grouped by year
summary_stats = df_factory.groupby('Year')[['TFR', 'F_CL_1013', 'M_CL_1013', 'IMR']].agg(['mean', 'std']).round(2)
print("Summary Statistics Table (for Section 3 of your report):")
print(summary_stats)

### **3.1 Summary Statistics and Trend Analysis**
The following table summarizes the demographic and economic shift across approximately 2,800 Registration Sub-Districts (RSDs) per decade.

| Variable | 1851 Mean | 1881 Mean | Trend |
| :--- | :--- | :--- | :--- |
| **TFR** | 4.47 | 4.63 | Slight increase / Stability |
| **Female Child Labor (10-13)** | 8.63% | 4.34% | **-50% Decline** |
| **Male Child Labor (10-13)** | 14.59% | 7.22% | **-50% Decline** |
| **Infant Mortality (IMR)** | 123.95 | 118.71 | Very gradual decline |

**Key Observations:**
1. **Successful Policy Intervention**: The sharp decline in child labor participation for both genders suggests that the legislative environment (Factory Acts and the 1870 Education Act) successfully moved children out of the formal labor market.
2. **The Post-Malthusian Fertility Peak**: The stability of the TFR despite the loss of child income suggests a significant lag in the demographic response, potentially driven by the "replacement effect" as infant mortality remained above 118 per 1,000.

In [25]:
# Identify textile-heavy districts based on the 1851 baseline
textile_districts = df_factory[(df_factory['Year'] == 1851) & (df_factory['F_TEX'] > df_factory['F_TEX'].median())]['REGDIST'].unique()

# Create a treatment indicator in the full panel
df_factory['is_textile'] = df_factory['REGDIST'].isin(textile_districts).astype(int)

In [26]:
# Plot average TFR trends for treatment and control groups
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_factory, x='Year', y='TFR', hue='is_textile', marker='o')
plt.title('TFR Trends: Textile vs. Non-Textile Districts (1851-1881)')
plt.ylabel('Total Fertility Rate')
plt.grid(True)
plt.show()

### **Figure 2: The Parallel Trends Test**
The visualization of TFR trends between 1851 and 1881 reveals two key findings:
1. **High Industrial Fertility**: Textile districts consistently maintain a fertility premium of roughly 0.7 children per woman over the control group.
2. **Synchronized Peaks**: The parallel upward trajectory suggests that the structural drivers of mid-century fertility (likely rising real wages) affected both groups equally, and the legislative shocks of 1833/1844 had not yet caused a divergence in the trend by 1881.

In [27]:
import statsmodels.formula.api as smf

# Simple model: TFR as a function of Child Labor and mortality
model = smf.ols('TFR ~ F_CL_1013 + IMR + C(Year)', data=df_factory).fit()
print(model.summary())

### **4. Regression Analysis: Determinants of Mid-Victorian Fertility**

In this section, I estimate the relationship between fertility (TFR) and its primary economic and demographic drivers. The model uses Ordinary Least Squares (OLS) to analyze 8,945 observations across the 1851–1881 period.

#### **4.1 Baseline Model Results**

| Variable | Coefficient | Std. Error | t-statistic | P-value |
| :--- | :--- | :--- | :--- | :--- |
| **Intercept** | 4.1517 | 0.029 | 140.865 | 0.000 |
| **Child Labor (F_CL_1013)** | 0.0036 | 0.001 | 2.804 | 0.005 |
| **Infant Mortality (IMR)** | 0.0023 | 0.000 | 10.878 | 0.000 |
| **Year: 1861** | 0.0447 | 0.019 | 2.380 | 0.017 |
| **Year: 1871** | 0.0816 | 0.032 | 2.549 | 0.011 |
| **Year: 1881** | 0.1835 | 0.020 | 9.389 | 0.000 |

* **R-squared:** 0.025
* **Durbin-Watson:** 0.916

#### **4.2 Demographic Interpretation**

**1. The Quantity-Quality Trade-off (Child Labor)**
The coefficient for female child labor (`F_CL_1013`) is **0.0036** and is statistically significant ($p = 0.005$). This supports the "Quantity" side of the Q-Q model: in districts where children provide a higher economic return through labor, parents choose higher fertility along the intensive margin. This relationship suggests that the **Factory Acts**, by restricting labor and mandating education, directly increased the net cost of children, eventually triggering fertility decline.

**2. Mortality and Replacement Fertility**
The `IMR` coefficient of **0.0023** is highly significant ($p < 0.001$). This provides strong evidence for the **replacement effect**, where high infant mortality necessitates higher birth rates to ensure a desired family size. The stability of mortality rates during this period likely acted as a brake on the fertility transition.

**3. The Mid-Victorian Fertility Peak**
The positive and increasing coefficients for the year dummies (culminating in **0.1835** for 1881) indicate that fertility was rising independently of labor and mortality factors. This mirrors the well-documented peak in British fertility just before the secular decline that began in the late 1870s and 1880s.

#### **4.3 Econometric Diagnostics**
* **Model Fit**: The low R-squared (0.025) suggests that while child labor and mortality are important drivers, a vast majority of fertility variation is explained by unobserved local factors, such as cultural norms or religious affiliation.
* **Autocorrelation**: The Durbin-Watson statistic of **0.916** indicates positive serial correlation in the residuals. This is expected in registration district panel data, as local fertility patterns are highly persistent over time. Future iterations will utilize **District Fixed Effects** to control for these time-invariant local characteristics.

### **5. Results & Discussion**
* **Figure 1**: Spatial distribution of TFR vs. Textile Industry in 1851.
* **Table 1**: Baseline regression of the "Textile Effect" on Fertility.
* **Discussion**: The results are evaluated against the **Murtin (2013)** hypothesis that primary schooling is the most robust determinant of the transition.

In [28]:
### 6. Parallel Trends Assessment
# Prepare DiD data using modular utility (identifies treatment based on baseline median)
df_panel = econometrics.prepare_did_sample(df_panel, 'F_TEX', 1851)

# Plot Trends
plotting.plot_parallel_trends(df_panel, 'Year', 'TFR', 'Treatment_Group')
plt.show()

print("Interpretation: If lines are parallel between waves, the DiD assumption is likely satisfied.")

In [14]:
### 7. Difference-in-Differences (DiD) Regression
# Formula: TFR ~ treat_dummy * Year + controls (IMR, Teacher density)
formula = "TFR ~ treat_dummy * C(Year) + IMR + SC6"
results = econometrics.run_clustered_ols(df_panel, formula, 'REGDIST')

print("Difference-in-Differences Estimation Results:")
print(results.summary())

In [15]:
### 8. Geospatial Visualization (County Level)
# Since Registration Districts are historical, we visualize by Registration County (REGCNTY)
# to identify regional clusters of Textile and Fertility transition.

county_data = df_panel[df_panel['Year'] == 1851].groupby('REGCNTY')[['TFR', 'F_TEX']].mean().reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Plotting Textile Intensity by County
sns.barplot(data=county_data.sort_values('F_TEX', ascending=False).head(15), 
            y='REGCNTY', x='F_TEX', ax=ax1, palette='viridis')
ax1.set_title('Top 15 Textile-Intensive Counties (1851)')
ax1.set_xlabel('Mean Female Textile Worker %')

# Plotting TFR by County to check visual correlation
sns.barplot(data=county_data.sort_values('TFR', ascending=False).head(15), 
            y='REGCNTY', x='TFR', ax=ax2, palette='magma')
ax2.set_title('Top 15 Highest Fertility Counties (1851)')
ax2.set_xlabel('Mean TFR')

plt.tight_layout()
plt.show()

print("Note: In a full GIS setup, we would join this to a .geojson shapefile of historical Britain.")

In [29]:
### 13. Model Diagnostics
# Generate standardized diagnostics using modular utility
corr_vars = ['TFR', 'treat_dummy', 'IMR', 'SC6']
plotting.plot_diagnostics(results, df_panel, corr_vars)
plt.show()

# VIF Assessment
vif_results = econometrics.calculate_vif(df_panel, ['treat_dummy', 'IMR', 'SC6'])
print("--- Variance Inflation Factors (VIF) ---")
print(vif_results)